# Explore the warehouse

Start the local stack, then run `source ops/local/env.sh` before launching Jupyter.
For a remote warehouse, set `MDP_ANALYST_URL` to your individual login through the Fly proxy.
Keep credentials out of this notebook. Install `psycopg[binary]`, `duckdb`, `pandas` and `matplotlib`.

Create a snapshot from the repository root:
`pnpm --dir control mdp warehouse snapshot --schemas marts --out ./warehouse.duckdb`
Set `MDP_SNAPSHOT_PATH` to that file if Jupyter starts in another directory.


In [ ]:
import os
import psycopg
import duckdb
import pandas as pd

with psycopg.connect(os.environ["MDP_ANALYST_URL"]) as connection:
    connection.execute("SET TRANSACTION READ ONLY")
    connection.execute("SET LOCAL statement_timeout = '5s'")
    with connection.cursor() as cursor:
        cursor.execute("SELECT to_regclass('catalog.relations')")
        catalog_exists = cursor.fetchone()[0] is not None
        query = "SELECT * FROM catalog.relations LIMIT 100" if catalog_exists else (
            "SELECT table_schema, table_name FROM information_schema.tables "
            "WHERE table_schema IN ('marts', 'staging', 'intermediate') LIMIT 100"
        )
        cursor.execute(query)
        catalog = pd.DataFrame(cursor.fetchall(), columns=[c.name for c in cursor.description])
catalog


## Save to your sandbox
Run this cell to copy a small mart with its rights columns, then open Sandbox.


In [ ]:
from psycopg import sql
with psycopg.connect(os.environ["MDP_ANALYST_URL"]) as connection:
    own = connection.execute("SELECT schema_name FROM catalog.sandbox_status WHERE owner_role=current_user ORDER BY schema_name LIMIT 1").fetchone()
    if own is None:
        raise ValueError("No owned sandbox. Ask an operator to create your individual login, then reconnect.")
    connection.execute(sql.SQL("CREATE TABLE IF NOT EXISTS {}.notebook_chart AS SELECT * FROM marts.mart_shazam_chart_daily LIMIT 100").format(sql.Identifier(own[0])))
    print(f"Saved {own[0]}.notebook_chart. Run mdp warehouse sandbox status and include this schema in your snapshot.")


In [ ]:
snapshot = duckdb.connect(os.environ.get("MDP_SNAPSHOT_PATH", "warehouse.duckdb"), read_only=True)
snapshot.sql("SELECT manifest FROM _mdp_snapshot").show()
chart = snapshot.sql("SELECT * FROM marts.mart_shazam_chart_daily").df()
chart.head()


Keep `learning_eligible`, `resale_permitted`, `source_keys` and label columns when saving rows.
False or unknown rights never become true because an analysis copies them.
Pseudonyms stay pseudonyms. Check catalog tenant labels before combining inputs.
The snapshot stamp describes each relation's own build; it does not promise a single build across all relations.


In [ ]:
import matplotlib.pyplot as plt

counts = chart.groupby("chart_date").size().sort_index()
ax = counts.plot(kind="bar", title="Observed chart entries by day", figsize=(9, 4))
ax.set_xlabel("Chart date")
ax.set_ylabel("Entries")
plt.tight_layout()


In [ ]:
snapshot.close()
